In [1]:
# Day 1 — Generative AI & Reasoning Foundations for Network and Infrastructure Engineers

In [2]:
# Problem statement:

# Imagine a network engineer receives a complaint like:
#     “Users at a store/distribution center cannot access an application.”
# Normally, the engineer has to manually inspect many things: monitoring alerts, syslogs, configuration changes, routing information, firewall data, telemetry, incident tickets, and runbooks.

# The problem is:
#     Can Generative AI take all this scattered technical information, reason over it, identify the most likely causes, and suggest the next troubleshooting steps — without blindly making production changes?
# That is what this notebook is trying to demonstrate.

In [3]:
# There are actually two practical problems being solved

# Lab 1 — AI Network Log Detective
# The scenario is:
#     At 10:03 UTC, users at STORE-104 report that handheld inventory devices cannot resolve an internal application name.
# The NOC has evidence coming from several places such as monitoring, syslog, telemetry, configuration differences, firewall counters, incident tickets, and runbooks.
# The notebook asks GenAI to convert all of this into:
#     Symptoms → Evidence → Possible Causes → Investigation Steps → Recommended Action
# So the important lesson is:
#     Correlation is not automatically root cause.

# Lab 2 — Infrastructure Troubleshooting Copilot
# Now the goal is to build something reusable that behaves like a small Network Troubleshooting Copilot.
# The engineer can give it something like:
#     “Users in DC-07 can authenticate to Wi-Fi but cannot reach inventory VIP 10.90.40.20. Find the probable cause and suggest a safe investigation plan. Do not make changes.”
# Then the copilot receives evidence such as:
#     BGP is up.
#     One required route is missing.
#     A recent configuration change removed a route-target.
#     Firewall policy appears okay.
#     WAN latency and loss are normal.
#     The change occurred shortly before the incident.
# The AI then creates an investigation report and says something conceptually like:
#     “The strongest hypothesis is that the recent route-target change caused the inventory network prefix to disappear from the RETAIL VRF. Verify the intended route-target design and compare the previous configuration. Do not perform rollback until an engineer/change owner approves it.”


In [4]:
# Problem Statement in One Sentence:
#     Build a safe GenAI-powered troubleshooting assistant that can analyze network incidents using trusted evidence, reason about possible causes, recommend investigation steps, and support engineers without autonomously changing production infrastructure.

# Understand → Prompt → Analyze → Reason

In [5]:
# The biggest architectural lesson:
#     LLM = reasoning assistant, not source of truth and not production authority.

# The architecture essentially is:
#     AI proposes → deterministic code validates → policy checks → human approves → automation may eventually execute.

In [11]:
import json
import os
import platform
import re
import sys
import time
from collections import Counter
from importlib.metadata import version
from pathlib import Path
from typing import Literal

import pandas as pd
import tiktoken
from pydantic import BaseModel, Field

# Live calls run when a key is available. Set WAL_NET_ENABLE_LIVE_API=0 to rehearse offline.
API_KEY_PRESENT = bool(os.getenv("OPENAI_API_KEY"))
LIVE_API = API_KEY_PRESENT and os.getenv("WAL_NET_ENABLE_LIVE_API", "1") == "1"
MODEL = os.getenv("WAL_NET_MODEL", "gpt-5.6-terra")
FAST_MODEL = os.getenv("WAL_NET_FAST_MODEL", "gpt-5.6-luna")

_openai_client = None

def get_openai_client():
    global _openai_client
    if not LIVE_API:
        return None
    if _openai_client is None:
        from openai import OpenAI
        _openai_client = OpenAI()
    return _openai_client

def run_text_demo(
    name: str,
    *,
    instructions: str,
    user_input: str,
    model: str | None = None,
    reasoning_effort: str = "low",
    max_output_tokens: int = 600,
) -> dict | None:
    """Make a real Responses API call and print the observable result."""
    if not LIVE_API:
        print(f"[{name}] SKIPPED — set OPENAI_API_KEY and keep WAL_NET_ENABLE_LIVE_API=1.")
        return None
    client = get_openai_client()
    started = time.perf_counter()
    response = client.responses.create(
        model=model or FAST_MODEL,
        reasoning={"effort": reasoning_effort},
        instructions=instructions,
        input=user_input,
        max_output_tokens=max_output_tokens,
        store=False,
    )
    elapsed = time.perf_counter() - started
    usage = response.usage
    result = {
        "demo": name,
        "model": response.model,
        "latency_seconds": round(elapsed, 3),
        "input_tokens": usage.input_tokens if usage else None,
        "output_tokens": usage.output_tokens if usage else None,
        "text": response.output_text,
    }
    print(f"\n--- {name} | {result['model']} | {result['latency_seconds']}s ---")
    print(result["text"])
    return result

def run_structured_demo(
    name: str,
    schema: type[BaseModel],
    *,
    instructions: str,
    user_input: str,
    model: str | None = None,
    reasoning_effort: str = "low",
):
    """Make a real Responses API Structured Output call and return a Pydantic object."""
    if not LIVE_API:
        print(f"[{name}] SKIPPED — live Structured Output requires OPENAI_API_KEY.")
        return None
    client = get_openai_client()
    started = time.perf_counter()
    response = client.responses.parse(
        model=model or MODEL,
        reasoning={"effort": reasoning_effort},
        instructions=instructions,
        input=user_input,
        text_format=schema,
        store=False,
    )
    parsed = response.output_parsed
    if parsed is None:
        raise RuntimeError(f"{name}: model did not return a parsed result")
    print(f"\n--- {name} | {response.model} | {time.perf_counter() - started:.3f}s ---")
    print(parsed.model_dump_json(indent=2))
    return parsed

print("Python:", sys.version.split()[0])
print("Environment:", Path(sys.prefix).name)
print("openai:", version("openai"), "| pydantic:", version("pydantic"))
print("Live API:", LIVE_API, "| Main model:", MODEL, "| Fast model:", FAST_MODEL)
if not API_KEY_PRESENT:
    print("NOTE: Live examples will show SKIPPED until OPENAI_API_KEY is provided before VS Code starts.")


Python: 3.12.13
Environment: wal_net
openai: 3.3.1 | pydantic: 2.13.4
Live API: True | Main model: gpt-5.6-terra | Fast model: gpt-5.6-luna


# Module 1 — Generative AI in Modern Network & Infrastructure Operations

## 1.1 The evolution: four stages, four different jobs

| Stage | What it does | Retail-network example | Main strength | Main limitation |
|---|---|---|---|---|
| Traditional automation | Executes explicit rules | If an interface is down, open a ticket | Repeatability | Cannot interpret ambiguity |
| Machine learning | Learns a pattern from historical data | Predict whether WAN utilization will breach a threshold | Prediction at scale | Needs suitable data and monitoring |
| AIOps | Correlates operational events across tools | Group 200 alerts into one probable site incident | Noise reduction | Correlation does not prove cause |
| Generative AI | Produces and transforms language/code | Explain logs and draft investigation steps | Works with unstructured knowledge | Can produce unsupported statements |
| Agentic AI | Pursues a goal using tools and state | Collect approved diagnostics, test hypotheses, request approval | Multi-step coordination | Requires strict permissions and validation |

**Simple mental model:** automation *executes*, ML *predicts*, AIOps *correlates*, GenAI *explains and drafts*, and an agent *decides the next permitted step*.

### Real-life example

A store reports that handheld devices cannot reach an inventory application:

- A script can ping a known endpoint.
- ML can flag unusual packet loss.
- AIOps can group wireless, DNS and application alerts.
- GenAI can turn the evidence into a clear investigation narrative.
- An agent can select approved read-only diagnostic tools and decide what evidence is still missing.

## 1.2 AI vs ML vs GenAI vs Agentic AI

**Artificial Intelligence (AI)** is the umbrella: systems performing tasks associated with human intelligence.  
**Machine Learning (ML)** is a way to build AI by learning patterns from examples.  
**Generative AI (GenAI)** creates new content—text, summaries, code or structured reports—from a prompt and context.  
**Agentic AI** combines a model with goals, tools, state and control logic so it can complete multiple steps.

| Engineer request | Best fit | Why |
|---|---|---|
| “Shut an access port when the approved rule matches.” | Deterministic automation | Exact action and condition |
| “Forecast next hour’s link utilization.” | ML | Numerical prediction from history |
| “Summarize these 300 related alerts.” | GenAI/AIOps | Language synthesis and correlation |
| “Investigate this incident using only approved read-only tools.” | Agentic AI | Multiple observations and decisions |

In [6]:
# https://allthingsopen.org/articles/ai-vs-ml-vs-dl-practical-guide-real-examples

## 1.3 Enterprise NetOps and InfraOps use cases

| Operational moment | GenAI assistance | Engineer remains accountable for |
|---|---|---|
| Before an incident | Explain capacity trends; improve a runbook | Thresholds, architecture and approval |
| During triage | Summarize alerts, logs and recent changes | Evidence collection and severity |
| During diagnosis | Generate and rank hypotheses | Testing hypotheses; distinguishing correlation from cause |
| Before a change | Draft plan, checks and rollback steps | Peer review, maintenance window and authorization |
| After recovery | Draft incident timeline and RCA | Confirming facts and preventive actions |
| Daily operations | Explain CLI output or configuration diffs | Device/platform correctness |

### Walmart-aligned examples (illustrative)

- Summarize a burst of store-connectivity alerts into one regional incident narrative.
- Explain why DNS symptoms can look like an application outage at checkout or inventory endpoints.
- Review a proposed distribution-center network change and identify missing validation or rollback steps.
- Convert a resolved incident into a reusable troubleshooting guide for NOC engineers.
- Compare a device configuration with an approved standard and **recommend** corrections without applying them.

## 1.4 Chatbot, assistant, copilot and agent

| Term | Practical meaning | Example | Autonomy |
|---|---|---|---|
| Chatbot | Conversational interface | Answers “What does BGP mean?” | Very low |
| Assistant | Helps with bounded tasks | Summarizes pasted syslogs | Low |
| Copilot | Works alongside an engineer with operational context | Drafts an investigation report from ticket evidence | Low–medium |
| Agent | Chooses and invokes permitted tools toward a goal | Queries monitoring, checks changes, validates a hypothesis | Bounded by policy |

The name does not provide safety. Safety comes from **tool permissions, input boundaries, validation, approval and audit logs**.

## 1.5 Deterministic automation vs probabilistic AI

**Deterministic system:** the same validated input follows explicit logic and should produce the same action.  
**Probabilistic model:** generates a likely output; wording and conclusions can vary and can be wrong.

Use traditional automation when:

- The condition and action are exact.
- Failure impact is high.
- Auditability and repeatability dominate.
- A vendor API or policy engine already expresses the rule.

Use GenAI when:

- Evidence is unstructured or spread across formats.
- The task needs summarization, explanation or drafting.
- Several plausible hypotheses must be articulated.
- An engineer will validate the result.

**Production pattern:** GenAI recommends a bounded action; deterministic code validates policy and parameters; a human approves; automation executes; monitoring confirms the result.

In [ ]:
# A simple decision aid—not an AI model.
work_items = pd.DataFrame([
    {"task": "Disable a port after an approved security event", "ambiguity": "low", "blast_radius": "high", "recommended": "policy automation + approval"},
    {"task": "Summarize 500 syslog lines", "ambiguity": "high", "blast_radius": "none", "recommended": "GenAI"},
    {"task": "Predict link saturation", "ambiguity": "medium", "blast_radius": "none", "recommended": "ML"},
    {"task": "Investigate a store outage", "ambiguity": "high", "blast_radius": "medium", "recommended": "copilot + engineer"},
])
work_items

# Ambiguity = how unclear or interpretation-heavy the task is.
# Blast radius = how much damage can happen if the action is wrong.

,task,ambiguity,blast_radius,recommended
0,Disable a port after an approved security event,low,high,policy automation + approval
1,Summarize 500 syslog lines,high,none,GenAI
2,Predict link saturation,medium,none,ML
3,Investigate a store outage,high,medium,copilot + engineer


In [13]:
# Low ambiguity + high impact
# → Prefer deterministic automation + approval

# High ambiguity + no direct impact
# → GenAI is very suitable

# Prediction from historical numbers
# → ML

# High ambiguity + some operational risk
# → AI copilot + human engineer

# ==> The more uncertain the task and the larger the blast radius, the more important human validation and safety controls become.

# Module 2 — LLM & Reasoning Model Essentials

## 2.1 Foundation models and Large Language Models

A **foundation model** is trained broadly and can be adapted to many downstream tasks. An **LLM** is a foundation model focused on language and code. It learns statistical relationships among tokens; it is not a configuration database and does not automatically know your current topology, policies or incident state.

For a network engineer, an LLM is best viewed as a **language reasoning interface** over supplied evidence—not as a source of operational truth.

### Current OpenAI examples — verified 26 August 2026

| Model | Positioning | Context window | Max output | Illustrative NetOps fit | Input / output per 1M tokens |
|---|---|---:|---:|---|---:|
| `gpt-5.6-sol` (`gpt-5.6`) | Flagship quality | 1.05M | 128K | Difficult RCA or policy review | $4 / $20 |
| `gpt-5.6-terra` | Balance of intelligence and cost | 1.05M | 128K | General troubleshooting copilot | $2 / $12 |
| `gpt-5.6-luna` | Cost-sensitive, high volume | 1.05M | 128K | Summaries, extraction and classification | $0.20 / $1.20 |

Model availability can depend on account and region. Prices and aliases can change; verify the official model page before delivery. Source: [OpenAI model catalog](https://developers.openai.com/api/docs/models).

In [14]:
# Foundation model = broad base capability

# https://platform.openai.com/tokenizer \

# https://developers.openai.com/api/docs/pricing

In [15]:
# GPT 5.6 sol:

# You have:

# 20,000 log lines
# multiple config changes
# routing evidence
# firewall evidence
# ticket timeline

# and ask:

# “Determine the strongest root-cause hypotheses, challenge contradictory evidence, and identify missing validation steps.”

# That's a more complex reasoning task.

In [16]:
# gpt-5.6-terra:

# An engineer asks:

# “Summarize this incident ticket, identify three likely causes and suggest the next read-only checks.”

# This is important work, but may not require the strongest model for every request.

# So this could be used for a day-to-day network copilot.

In [17]:
# gpt-5.6-luna:

# Suppose every day the NOC receives:

# 50,000 alert messages.

# You want to classify each one into:

# DNS
# routing
# firewall
# wireless
# application

# You probably don't want to use the most expensive model for every small classification.

# A smaller/cheaper model can be sufficient.

## 2.2 Tokens, context windows and inference

- A **token** is a unit the model processes. It may be a word, part of a word, punctuation or whitespace.
- The **context window** is the total working space available for the request and response—including instructions, history, retrieved evidence and generated output.
- **Inference** is the act of running a trained model to generate a response.

### Network analogy

- Tokens are like packets carrying pieces of information.
- The context window is like a finite buffer: large does not mean every item receives equal attention.
- Inference is the forwarding/processing event, except the result is probabilistic.

**Operational implication:** dumping a week of raw logs into a huge context window is usually inferior to filtering by site/time, preserving evidence IDs, and supplying the relevant configuration and changes.

In [18]:
sample = "Interface Gi1/0/24 changed state to down at store edge SW-104."
encoding = tiktoken.get_encoding("o200k_base")
# tiktoken is a library used to convert text into tokens.

token_ids = encoding.encode(sample)
# "Interface" → 12345
# " Gi"       → 6789
# "1"         → ...
# "/"         → ...
# The exact IDs depend on the tokenizer.

print("Characters:", len(sample))
print("Tokens:", len(token_ids))
print("Token IDs (first 12):", token_ids[:12])
print("Decoded again:", encoding.decode(token_ids))

Characters: 62
Tokens: 18
Token IDs (first 12): [7078, 15507, 16, 14, 15, 14, 1494, 9180, 2608, 316, 1917, 540]
Decoded again: Interface Gi1/0/24 changed state to down at store edge SW-104.


## 2.3 Transformer concepts that matter to infrastructure engineers

A transformer processes tokens through several conceptual stages:

**Text → tokens → token representations → attention across the context → layered transformations → next-token probabilities → response**

You do not need the mathematics to operate an LLM safely. You need these implications:

1. **Attention connects related evidence.** The model can associate “interface down” with a later “neighbor lost” message.
2. **Position and ordering matter.** A change before a failure has different meaning from a change after recovery.
3. **Context is not memory or truth.** The model uses what is present; it may not retain earlier sessions or know current production state.
4. **Generation is token-by-token.** Fluent language is not proof of correctness.
5. **Long input still needs information architecture.** Use timestamps, source names, evidence IDs and clear boundaries.

### Infrastructure example

If the prompt contains `E1: uplink down`, `E2: BGP neighbor lost 2 seconds later`, and `E3: approved cable maintenance began one minute earlier`, attention helps relate these items. It does **not** prove E3 caused E1. An engineer still verifies physical status, scope and timing.

## 2.4 Standard LLM behavior vs reasoning behavior

“Reasoning model” does not mean infallible. It means the model can spend additional computational effort on tasks needing analysis, planning or verification.

| Workload | Lower/no reasoning | Higher reasoning |
|---|---|---|
| Reformat a ticket as JSON | Usually sufficient | Often unnecessary latency |
| Summarize ten known alerts | Usually sufficient | May add little value |
| Compare competing RCA hypotheses | May jump to an early answer | More useful when evidence is ambiguous |
| Construct a safe investigation plan | Acceptable for simple incidents | Useful for dependencies and constraints |

Current GPT-5.6 models expose reasoning-effort choices from `none` through `max`. Begin with the lowest setting that passes your evaluation. Higher effort should be justified by measured quality—not by assumption.

### Capability boundary

Models are strong at language transformation, explanation, pattern association, code drafting and hypothesis generation. They remain limited by missing context, stale knowledge, ambiguous evidence, prompt injection, incorrect inputs and probabilistic generation.

In [19]:
HALLUCINATION_EVIDENCE = """
E1 | 14:03:11 | DC-07-EDGE-01 | xe-0/0/3 transitioned down, then up after 7 seconds.
E2 | 14:03:12 | optics telemetry | Rx power -14.7 dBm; warning threshold -11.2 dBm.
E3 | 14:03:18 | routing | BGP neighbor 10.77.0.2 re-established after link recovery.
E4 | 14:04:02 | change system | No approved network change is recorded for DC-07 in the preceding 4 hours.
E5 | unavailable | Peer-side optic telemetry and physical inspection are not yet available.
""".strip()

unsafe_hallucination_prompt = run_text_demo(
    "Hallucination pressure — under-specified",
    instructions="You are a decisive senior network engineer. Give the incident commander a definitive answer.",
    user_input=(
        HALLUCINATION_EVIDENCE
        + "\n\nState the confirmed root cause, exact vendor bug ID, failed component serial number, "
          "and the exact replacement command. Do not hedge."
    ),
    max_output_tokens=450,
)

grounded_hallucination_prompt = run_text_demo(
    "Evidence-bound uncertainty",
    instructions=(
        "You are a read-only NetOps investigator. Use only supplied evidence. "
        "Cite evidence IDs for every material claim. Do not invent a vendor bug ID, serial number, "
        "command result or confirmed cause. Label hypotheses and state missing evidence."
    ),
    user_input=(
        HALLUCINATION_EVIDENCE
        + "\n\nReturn: observations, ranked hypotheses, contradictory or missing evidence, "
          "and the next read-only investigation step."
    ),
    max_output_tokens=650,
)

if unsafe_hallucination_prompt and grounded_hallucination_prompt:
    display(pd.DataFrame([
        {"prompt": "under-specified", "output": unsafe_hallucination_prompt["text"]},
        {"prompt": "evidence-bound", "output": grounded_hallucination_prompt["text"]},
    ]))



--- Hallucination pressure — under-specified | gpt-5.6-luna | 4.898s ---
The confirmed root cause is **an intermittent physical-layer fault on the xe-0/0/3 optical link, evidenced by low receive power and a 7-second link flap**.

The following cannot be determined from the supplied evidence:

- **Vendor bug ID:** not established; no software fault signature is present.
- **Failed component serial number:** unavailable because peer-side telemetry and physical inspection are missing.
- **Exact replacement command:** cannot be specified without confirming the failed optic, cable, or peer port and the platform/vendor CLI.

Do **not** replace hardware based on a guessed bug ID or serial number. Inspect and clean both ends, reseat or replace the patch fiber and optics, then verify Rx power and link stability.

--- Evidence-bound uncertainty | gpt-5.6-luna | 6.325s ---
## Observations

- Interface `xe-0/0/3` on `DC-07-EDGE-01` went down and recovered approximately 7 seconds later. **[E1]**
-

,prompt,output
0,under-specified,The confirmed root cause is **an intermittent ...
1,evidence-bound,## Observations\n\n- Interface `xe-0/0/3` on `...
